# Prompting vs. Log-Probability Comparison (Multi-Model)

Compares prompting-based evaluation against log-probability-based evaluation across multiple instruction-tuned causal models, using a 25-pairs-per-phenomenon sample (justified by the sample-size stability results already computed for the main evaluation).

Mirrors the comparison in Sjons et al. (2026, Swe-BLiMP Table 3). Handles both local HuggingFace models and the OpenAI API model (GPT-5.4-nano) in one notebook.

**Models are auto-discovered** from your saved `results/autoregressive/` folder, no need to type model names/paths manually. Since prompting only works reliably on instruction-tuned models, edit `EXCLUDE_MODELS` below to skip base models rather than typing out a full config for each one.

In [4]:
import os
os.environ["HF_HOME"] = "/content/drive/MyDrive/Thesis/hf_cache"

In [5]:
# Setup
import gc
import json
import random
import time
from pathlib import Path

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

print(f"GPU available: {torch.cuda.is_available()}")

GPU available: True


In [6]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Config

In [7]:
AUTOREGRESSIVE_RESULTS_DIR = Path("/content/drive/MyDrive/Thesis/results/autoregressive")  # adjust to your actual path
PHENOMENA_DIR = Path("/content/drive/MyDrive/Thesis/data/phenomena")                        # adjust to your actual path
OUTPUT_DIR = Path("/content/drive/MyDrive/Thesis/results/prompting")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

N_PAIRS_PER_PHENOMENON = 25
RANDOM_SEED = 42

# Models to SKIP -- base (non-instruction-tuned) models generally fail to
# follow "answer only A or B" reliably, producing noisy, unparseable
# results. List substrings that appear in the model name / results
# filename; any match gets excluded from auto-discovery.
EXCLUDE_MODELS = [
    "gpt2",              # base GPT-2, not instruction-tuned
    "Krikri-8B-Base",    # explicitly a base model
    # add more substrings here as needed, e.g. "Meltemi", "EuroLLM", etc.
    # if you find they are base (not instruct) versions
]

from google.colab import userdata
API_KEY = userdata.get("OPENAI_API_KEY")

## Auto-discover models from your saved log-prob results\n\nScans `results/autoregressive/` for every `*_results.json` file, extracts the model name from inside each file, and builds the model list automatically. Anything matching `EXCLUDE_MODELS` is skipped and listed separately so you can double check nothing important was excluded by accident.

In [8]:
def discover_models(results_dir):
    discovered = []

    for result_file in sorted(results_dir.glob("*_results.json")):
        with open(result_file, encoding='utf-8') as f:
            data = json.load(f)
        model_name = data.get('model', result_file.stem)



        discovered.append({
            "backend": "local",
            "model_name": model_name,
            "existing_results_path": str(result_file),
        })

    return discovered


MODELS_TO_CHECK = discover_models(AUTOREGRESSIVE_RESULTS_DIR)

# Add GPT-5.4-nano manually -- it has no existing log-prob results file
# (API-only model, never evaluated via log-prob scoring)
MODELS_TO_CHECK.append({
    "backend": "openai",
    "model_name": "gpt-5.4-nano-2026-03-17",
    "existing_results_path": None,
})

print(f"Included ({len(MODELS_TO_CHECK)} models):")
for m in MODELS_TO_CHECK:
    print(f"  [{m['backend']:>6}] {m['model_name']}")

Included (9 models):
  [ local] Qwen/Qwen2.5-7B
  [ local] allenai/OLMo-2-1124-7B
  [ local] ilsp/Llama-Krikri-8B-Base
  [ local] ilsp/Meltemi-7B-v1
  [ local] lighteternal/gpt2-finetuned-greek
  [ local] meta-llama/Llama-3.1-8B
  [ local] openai-community/gpt2
  [ local] utter-project/EuroLLM-1.7B
  [openai] gpt-5.4-nano-2026-03-17


**Check the printed lists above before continuing** — confirm nothing you actually wanted got excluded, and that nothing you meant to exclude (e.g. an undiscovered base model) slipped through. Adjust `EXCLUDE_MODELS` and re-run the cell above if needed.

## Data loading

In [9]:
def load_phenomenon_pairs(phenomenon_path):
    gram_file = phenomenon_path / "correct.txt"
    ungram_file = phenomenon_path / "incorrect.txt"
    with open(gram_file, 'r', encoding='utf-8') as f:
        gram_sentences = f.readlines()
    with open(ungram_file, 'r', encoding='utf-8') as f:
        ungram_sentences = f.readlines()
    return list(zip(gram_sentences, ungram_sentences))


def load_all_phenomena(data_dir):
    all_pairs = {}
    for folder in Path(data_dir).iterdir():
        if folder.is_dir():
            name = folder.name[5:] if folder.name.startswith("DONE_") else folder.name
            try:
                pairs = load_phenomenon_pairs(folder)
                all_pairs[name] = pairs
            except FileNotFoundError:
                pass
    return all_pairs


all_data = load_all_phenomena(PHENOMENA_DIR)
print(f"Total phenomena: {len(all_data)}")

Total phenomena: 6


## Sampling — same 25 pairs per phenomenon used for EVERY model, so results are directly comparable

In [10]:
def get_sample_pairs(all_data, n_per_phenomenon, seed):
    random.seed(seed)
    sampled = {}
    for phenomenon, pairs in all_data.items():
        sampled[phenomenon] = random.sample(pairs, min(n_per_phenomenon, len(pairs)))
    return sampled


sample_pairs = get_sample_pairs(all_data, N_PAIRS_PER_PHENOMENON, RANDOM_SEED)
for phen, pairs in sample_pairs.items():
    print(f"  {phen}: {len(pairs)} pairs sampled")

  noun_adjective_agreement: 25 pairs sampled
  aspect: 25 pairs sampled
  einai_agreement: 25 pairs sampled
  subject_verb_agreement: 25 pairs sampled
  negations: 25 pairs sampled
  case_selection: 25 pairs sampled


In [11]:
for f in sorted(OUTPUT_DIR.glob("*_prompting_comparison.json")):
    print(f.name, f.stat().st_size, "bytes")

gpt-5_4-nano-2026-03-17_prompting_comparison.json 58715 bytes


## Backends: local (HuggingFace) and OpenAI

In [12]:
def ask_local(tokenizer, model, sentence_a, sentence_b):
    prompt = (
        f"Which sentence is grammatically correct Modern Greek?\n"
        f"A: {sentence_a}\n"
        f"B: {sentence_b}\n"
        f"Answer with only the letter A or B."
    )
    if getattr(tokenizer, "chat_template", None):
        messages = [{"role": "user", "content": prompt}]
        encoded = tokenizer.apply_chat_template(
            messages, add_generation_prompt=True, return_tensors="pt"
        )
        # apply_chat_template can return either a raw tensor OR a
        # BatchEncoding/dict, depending on the tokenizer -- handle both
        if hasattr(encoded, "input_ids"):
            input_ids = encoded.input_ids.to(model.device)
        elif isinstance(encoded, dict):
            input_ids = encoded["input_ids"].to(model.device)
        else:
            input_ids = encoded.to(model.device)
    else:
        plain_prompt = prompt + "\nAnswer:"
        input_ids = tokenizer(plain_prompt, return_tensors="pt").input_ids.to(model.device)

    with torch.no_grad():
        output = model.generate(input_ids, max_new_tokens=5, do_sample=False)

    generated = tokenizer.decode(output[0][input_ids.shape[1]:], skip_special_tokens=True).strip()
    for char in generated.upper():
        if char in ('A', 'B'):
            return char
    return None


def ask_openai(client, model_name, sentence_a, sentence_b, max_retries=3):
    prompt = (
        f"Which sentence is grammatically correct Modern Greek?\n"
        f"A: {sentence_a}\n"
        f"B: {sentence_b}\n"
        f"Answer with only the letter A or B."
    )
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=model_name,
                messages=[{"role": "user", "content": prompt}],
                temperature=0,
                max_completion_tokens=15,
            )
            text = response.choices[0].message.content.strip().upper()
            for char in text:
                if char in ('A', 'B'):
                    return char
            return None
        except Exception as e:
            print(f"    API error (attempt {attempt + 1}/{max_retries}): {e}")
            time.sleep(2 ** attempt)
    return None

## Helper: look up the log-prob verdict for a pair from existing results

In [13]:
def get_logprob_correctness(existing_results_path, phenomenon, gram, ungram):
    """Look up whether the log-prob method got this specific pair right,
    from the existing full results file. Returns None if not found
    (e.g. for models with no existing log-prob results, like API-only models)."""
    if existing_results_path is None:
        return None
    with open(existing_results_path, encoding='utf-8') as f:
        existing = json.load(f)
    for pair in existing['per_phenomenon'][phenomenon]['pairs']:
        if pair['grammatical'].strip() == gram.strip() and pair['ungrammatical'].strip() == ungram.strip():
            return pair['correct']
    return None

## Main per-model evaluation function

In [14]:
def evaluate_model(model_config, sample_pairs):
    model_name = model_config["model_name"]
    backend = model_config["backend"]
    existing_path = model_config["existing_results_path"]

    safe_name = model_name.replace('/', '_').replace('.', '_')
    output_path = OUTPUT_DIR / f"{safe_name}_prompting_comparison.json"

    if output_path.exists():
        print(f"Skipping {model_name}: already done.")
        return

    print(f"\n{'='*70}\nEvaluating: {model_name} ({backend})\n{'='*70}")

    tokenizer, model, client = None, None, None
    try:
        if backend == "local":
            tokenizer = AutoTokenizer.from_pretrained(model_name)
            model = AutoModelForCausalLM.from_pretrained(
                model_name, torch_dtype=torch.float16, device_map='auto'
            )
            model.eval()
        elif backend == "openai":
            if not API_KEY:
                raise RuntimeError("OPENAI_API_KEY environment variable not set.")
            from openai import OpenAI
            client = OpenAI(api_key=API_KEY)

        comparison_rows = []
        random.seed(RANDOM_SEED)

        for phenomenon, pairs in sample_pairs.items():
            print(f"\n{phenomenon} ({len(pairs)} pairs)")
            for gram, ungram in pairs:
                if random.random() < 0.5:
                    sentence_a, sentence_b, correct_letter = gram, ungram, 'A'
                else:
                    sentence_a, sentence_b, correct_letter = ungram, gram, 'B'

                if backend == "local":
                    answer = ask_local(tokenizer, model, sentence_a, sentence_b)
                else:
                    answer = ask_openai(client, model_name, sentence_a, sentence_b)

                prompting_correct = (answer == correct_letter)
                logprob_correct = get_logprob_correctness(existing_path, phenomenon, gram, ungram)

                comparison_rows.append({
                    'phenomenon': phenomenon,
                    'grammatical': gram.strip(),
                    'ungrammatical': ungram.strip(),
                    'model_answer': answer,
                    'prompting_correct': prompting_correct,
                    'logprob_correct': logprob_correct,
                    'methods_agree': (logprob_correct == prompting_correct) if logprob_correct is not None else None,
                })

        prompting_acc = sum(r['prompting_correct'] for r in comparison_rows) / len(comparison_rows)
        rows_with_logprob = [r for r in comparison_rows if r['logprob_correct'] is not None]
        logprob_acc = (sum(r['logprob_correct'] for r in rows_with_logprob) / len(rows_with_logprob)
                        if rows_with_logprob else None)
        agreement = (sum(r['methods_agree'] for r in rows_with_logprob) / len(rows_with_logprob)
                     if rows_with_logprob else None)

        print(f"\nPrompting accuracy: {prompting_acc:.1%}")
        if logprob_acc is not None:
            print(f"Log-prob accuracy:  {logprob_acc:.1%}")
            print(f"Method agreement:   {agreement:.1%}")
        else:
            print("(No existing log-prob results to compare against for this model.)")

        output = {
            'model': model_name,
            'backend': backend,
            'n_pairs_per_phenomenon': N_PAIRS_PER_PHENOMENON,
            'prompting_accuracy': prompting_acc,
            'logprob_accuracy': logprob_acc,
            'method_agreement': agreement,
            'pairs': comparison_rows,
        }

        with open(output_path, 'w', encoding='utf-8') as f:
            json.dump(output, f, ensure_ascii=False, indent=2)
            f.flush()
            os.fsync(f.fileno())
        print(f"Saved to {output_path}")

    except Exception as e:
        import traceback
        print(f"FAILED on {model_name}: {type(e).__name__}: {e}")
        traceback.print_exc()

    finally:
        try:
            del model, tokenizer
        except NameError:
            pass
        gc.collect()
        torch.cuda.empty_cache()

## Run the loop across all auto-discovered models

In [15]:
for model_config in MODELS_TO_CHECK:
    evaluate_model(model_config, sample_pairs)

print("\n" + "="*70)
print("ALL MODELS DONE")
print("="*70)


Evaluating: Qwen/Qwen2.5-7B (local)


config.json:   0%|          | 0.00/686 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.23k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/219 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.31GB            

model.safetensors: downloading bytes:           |  0.00B            

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]


noun_adjective_agreement (25 pairs)

aspect (25 pairs)

einai_agreement (25 pairs)

subject_verb_agreement (25 pairs)

negations (25 pairs)

case_selection (25 pairs)

Prompting accuracy: 48.7%
Log-prob accuracy:  82.0%
Method agreement:   44.0%
Saved to /content/drive/MyDrive/Thesis/results/prompting/utter-project_EuroLLM-1_7B_prompting_comparison.json
Skipping gpt-5.4-nano-2026-03-17: already done.

ALL MODELS DONE


## Quick summary table across all completed comparisons

In [16]:
import pandas as pd

summary_rows = []
for result_file in OUTPUT_DIR.glob("*_prompting_comparison.json"):
    with open(result_file, encoding='utf-8') as f:
        data = json.load(f)
    summary_rows.append({
        'Model': data['model'],
        'Backend': data['backend'],
        'Prompting Acc.': data['prompting_accuracy'],
        'Log-prob Acc.': data['logprob_accuracy'],
        'Agreement': data['method_agreement'],
    })

if summary_rows:
    df = pd.DataFrame(summary_rows)
    print(df.to_string(index=False))
else:
    print("No results found yet.")

                            Model Backend  Prompting Acc.  Log-prob Acc.  Agreement
          gpt-5.4-nano-2026-03-17  openai        0.933333            NaN        NaN
                  Qwen/Qwen2.5-7B   local        0.753333       0.880000   0.673333
           allenai/OLMo-2-1124-7B   local        0.493333       0.760000   0.440000
        ilsp/Llama-Krikri-8B-Base   local        0.406667       0.866667   0.380000
               ilsp/Meltemi-7B-v1   local        0.493333       0.906667   0.493333
lighteternal/gpt2-finetuned-greek   local        0.020000       0.793333   0.226667
          meta-llama/Llama-3.1-8B   local        0.700000       0.813333   0.633333
            openai-community/gpt2   local        0.000000       0.593333   0.406667
       utter-project/EuroLLM-1.7B   local        0.486667       0.820000   0.440000


In [17]:
print(f"Path: {AUTOREGRESSIVE_RESULTS_DIR}")
print(f"Exists: {AUTOREGRESSIVE_RESULTS_DIR.exists()}")

if AUTOREGRESSIVE_RESULTS_DIR.exists():
    all_files = list(AUTOREGRESSIVE_RESULTS_DIR.glob("*"))
    print(f"\nAll files in folder ({len(all_files)}):")
    for f in all_files:
        print(f"  {f.name}")

Path: /content/drive/MyDrive/Thesis/results/autoregressive
Exists: True

All files in folder (16):
  utter-project_EuroLLM-1.7B_results.json
  utter-project_EuroLLM-1.7B_stability.json
  openai-community_gpt2_results.json
  openai-community_gpt2_stability.json
  meta-llama_Llama-3.1-8B_results.json
  meta-llama_Llama-3.1-8B_stability.json
  ilsp_Meltemi-7B-v1_results.json
  ilsp_Meltemi-7B-v1_stability.json
  allenai_OLMo-2-1124-7B_results.json
  allenai_OLMo-2-1124-7B_stability.json
  Qwen_Qwen2.5-7B_results.json
  Qwen_Qwen2.5-7B_stability.json
  ilsp_Llama-Krikri-8B-Base_results.json
  ilsp_Llama-Krikri-8B-Base_stability.json
  lighteternal_gpt2-finetuned-greek_results.json
  lighteternal_gpt2-finetuned-greek_stability.json
